# TorchTPU Precision Control Tutorial

This tutorial demonstrates how to control the precision of matrix multiplication and convolution operations on TPU using `torch.tpu.precision`.

TPUs support different precision modes for `float32` operations, allowing you to trade off between performance and accuracy.

## Precision Modes

Specify the precision modes using the `torch.tpu.Precision` enum (See [XMU Floating Point Precision](https://docs.pytorch.org/xla/release/2.8/tutorials/precision_tutorial.html) for algorithm details):
- `Precision.DEFAULT`: The default precision. This is the fastest available precision that maintains reasonable accuracy (often `bfloat16` for the compute, but `float32` for accumulation).
- `Precision.HIGH`: Uses multiple `bfloat16` passes or valid hardware features to approximate `float32` precision with more accuracy than `Precision.DEFAULT` but slower.
- `Precision.HIGHEST`: Uses multiple `bfloat16` passes or valid hardware features to approximate or may use `float32` precision. This is the slowest but most accurate mode.


In [1]:
import time
import torch
from torch_tpu import api

api.tpu_device()

libtpu not found.


Successfully renamed PrivateUse1 backend to 'tpu'. Device: tpu
Registered Python module for 'tpu'.


Device type: tpu, Device index: default
Initializing TPU distributed runtime


device(type='tpu')

## Basic Usage

Use `torch.tpu.precision` as a context manager.

In [4]:
precision = torch.tpu.precision
Precision = torch.tpu.Precision


def run_matmul():
  size = (8, 2048, 2048)
  a = torch.zeros(size, device="tpu")
  b = torch.zeros(size, device="tpu")
  return torch.matmul(a, b).cpu()


for mode in [Precision.DEFAULT, Precision.HIGH, Precision.HIGHEST]:
  # Clear the cache to ensure that the compilation time is measured.
  torch.tpu._clear_cache()

  with precision(mode):
    start = time.time()
    run_matmul()
    print(f"{mode} time: {time.time() - start:.4f}s")

Precision.DEFAULT time: 0.1107s
Precision.HIGH time: 0.1644s
Precision.HIGHEST time: 0.2762s


## Accuracy Comparison

Let's compare the results against a CPU reference to see the difference in accuracy.

In [8]:
size = (8, 2048, 2048)
rand_a = torch.randn(size, device="tpu")
rand_b = torch.randn(size, device="tpu")

# CPU reference result
c_ref = torch.matmul(rand_a.cpu(), rand_b.cpu())

for mode in [Precision.DEFAULT, Precision.HIGH, Precision.HIGHEST]:
  with precision(mode):
    res = torch.matmul(rand_a, rand_b)
    diff = res.cpu() - c_ref
    print(f"Mode: {mode}")
    print(f"  Max Absolute Error: {diff.abs().max():.6e}")
    print(
        "  Mean Relative Error: "
        f" {(diff.abs() / (c_ref.abs() + 1e-8)).mean():.6e}\n"
    )

Mode: Precision.DEFAULT
  Max Absolute Error: 5.958481e-01
  Mean Relative Error:  3.568539e-02

Mode: Precision.HIGH
  Max Absolute Error: 3.295898e-03
  Mean Relative Error:  1.493241e-04

Mode: Precision.HIGHEST
  Max Absolute Error: 1.678467e-04
  Mean Relative Error:  4.096909e-06

